# Identificação de potenciais biomarcadores com Aprendizado de Máquina

---

In [1]:
import pandas as pd
import numpy as np
import glob
import os  
from joblib import load
import shap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from skbio.stats.composition import clr
from sklearn.metrics import (
    roc_auc_score, balanced_accuracy_score, accuracy_score, f1_score,
    confusion_matrix, classification_report, 
    ConfusionMatrixDisplay, RocCurveDisplay, average_precision_score, PrecisionRecallDisplay  
)
import matplotlib.pyplot as plt  
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

### Metataxonomia (Amplicon)

In [7]:
SEMENTE_ALEATORIA = 24
TAMANHO_TESTE = 0.15
PSEUDOCOUNT = 1
PASTA_MODELOS = 'modelos-amplicon/'
PASTA_PLOTS = 'modelos-amplicon/plots/' 

os.makedirs(PASTA_PLOTS, exist_ok=True)  

map_labels = {
    "controle": "Controle",
    "endo": "Endometriose",
    "fbm": "Fibromialgia",
    "cfs": "SFC",
    "ibs": "SII",
    "cpp": "DPC",
    "cpp_endo": "Endometriose + DPC"
}

data = pd.read_csv('taxonomia_meta-analise-int_corrigida.csv').rename({'Unnamed: 0': 'index'}, axis=1).set_index('index')
metadata = pd.read_csv('../02 - processamento/amplicon/intestinal/meta-analise-int/meta-analise-int_metadata.tsv', sep='\t').set_index("sample-id")

def ultimo_nivel(taxon):
    partes = taxon.split(';')
    preenchido = [p for p in partes if not p.endswith('__')]
    nivel = preenchido[-1] if preenchido else partes[-1]
    return nivel.split('__')[-1]

data["Condição"] = metadata.loc[data.index, "group"]


pares_separados = [
    ("Controle vs. FBM", data.loc[~data["Condição"].isin(['endo', 'ibs', 'cfs'])]),
    ("Controle vs. CFS", data.loc[~data["Condição"].isin(['endo', 'ibs', 'fbm'])]),
    ("Controle vs. IBS", data.loc[~data["Condição"].isin(['endo', 'fbm', 'cfs'])]),
    ("Controle vs. Endometriose", data.loc[~data["Condição"].isin(['ibs', 'fbm', 'cfs'])]),
]

for nome_par, dados_par in pares_separados:
    print("\n" + "="*70)
    print(f"{nome_par}")
    print("="*70)

    labels_originais = dados_par["Condição"].values  

    y = (labels_originais != "controle").astype(int)

    dados_par = dados_par.drop(columns=['Condição'])
    
    nomes_taxons = dados_par.columns
    
    ultimos_niveis = [ultimo_nivel(c) for c in nomes_taxons]
    
    print(f'Número de táxons inicial: {len(dados_par.columns)}')
    
    nomes_para_plotar = [map_labels[label] for label in labels_originais]
    
    dados_rel = dados_par.div(dados_par.sum(axis=1), axis=0)
    keep_taxa = (dados_rel >= 0.0001).sum(axis=0) >= (0.10 * dados_rel.shape[0])
    dados_par = dados_par.loc[:, keep_taxa]

    X = dados_par.values
    
    print(f'Número de táxons pós-filtro: {len(dados_par.columns)}')
    
    X_treino, X_teste, y_treino, y_teste = train_test_split(
        X, y, test_size=TAMANHO_TESTE, random_state=SEMENTE_ALEATORIA, stratify=y
    )

    X_treino_clr = clr(X_treino + PSEUDOCOUNT)
    X_teste_clr = clr(X_teste + PSEUDOCOUNT)
    
    grupos = np.unique(y_treino)
    g1 = X_treino_clr[y_treino == grupos[0]]
    g2 = X_treino_clr[y_treino == grupos[1]]
    
    pvals = []
    
    for i in range(X_treino_clr.shape[1]):
        try:
            stat, p = mannwhitneyu(g1[:, i], g2[:, i], alternative="two-sided")
        except ValueError:
            p = 1.0
        pvals.append(p)
        
    pvals = np.array(pvals)
    
    _, pvals_corr, _, _ = multipletests(pvals, method="fdr_bh")
    
    n_features_desejado = int(0.3 * X_treino_clr.shape[0])
    idx_top = np.argsort(pvals_corr)[:n_features_desejado]
    
    X_treino_sel = X_treino_clr[:, idx_top]
    X_teste_sel  = X_teste_clr[:, idx_top]
    
    dados_par = dados_par.iloc[:, idx_top]

    print(f'Número de táxons pós-seleção: {len(dados_par.columns)}')

    normalizador = StandardScaler()
    X_treino_norm = normalizador.fit_transform(X_treino_sel)
    X_teste_norm = normalizador.transform(X_teste_sel) 

    nome_base_arquivo = nome_par.replace(' vs. ', '_vs_')
    padrao_busca = f"{PASTA_MODELOS}{nome_base_arquivo}*.joblib"
    modelos_encontrados = glob.glob(padrao_busca)

    for file_path in modelos_encontrados:
        model_name = os.path.basename(file_path) 
        print(f"\n--- Avaliando: {model_name} ---")

        modelo = load(file_path)

        y_pred = modelo.predict(X_teste_norm)
        y_proba = modelo.predict_proba(X_teste_norm)

        print(f"  Acurácia balanceada: {balanced_accuracy_score(y_teste, y_pred):.3f}")
        print(f"  PR AUC: {average_precision_score(y_teste, y_proba[:, 1]):.3f}")
        print(f"  F1-score: {f1_score(y_teste, y_pred):.3f}")
        
        plt.figure(figsize=(7, 6))
        ax1 = plt.gca()

        ConfusionMatrixDisplay.from_estimator(
            modelo,
            X_teste_norm,
            y_teste,
            ax=ax1,
            display_labels=[
                map_labels["controle"],
                map_labels[np.unique(labels_originais[labels_originais != "controle"])[0]]
            ],
            cmap='RdPu'
        )

        ax1.set_xlabel('Predito', fontsize=12)
        ax1.set_ylabel('Verdadeiro', fontsize=12) 

        plt.tight_layout()

        save_path_cm = os.path.join(PASTA_PLOTS, model_name.replace('.joblib', '_confusion_matrix.png'))
        plt.savefig(save_path_cm, dpi=300, bbox_inches='tight')

        #plt.show()
        plt.close()

        plt.figure(figsize=(7, 6))
        ax2 = plt.gca()

        nome_controle = map_labels["controle"]
        nome_doenca = map_labels[np.unique(labels_originais[labels_originais != "controle"])[0]]

        PrecisionRecallDisplay.from_estimator(
            modelo,
            X_teste_norm,
            y_teste,
            ax=ax2,
            name=f'{nome_controle} vs {nome_doenca}',
            color='#C04ABC'
        )

        ax2.plot([0, 1], [sum(y_teste)/len(y_teste)]*2, 'k--', label='Baseline')

        ax2.set_xlabel('Sensibilidade', fontsize=12)
        ax2.set_ylabel('Precisão', fontsize=12)
        ax2.legend(loc='lower left')

        plt.tight_layout()

        save_path_pr = os.path.join(PASTA_PLOTS, model_name.replace('.joblib', '_precision_recall.png'))
        plt.savefig(save_path_pr, dpi=300, bbox_inches='tight')

        #plt.show()
        plt.close()
        
        if hasattr(modelo, "feature_importances_"):  
            explainer = shap.TreeExplainer(modelo)
        else:
            explainer = shap.Explainer(modelo, X_treino_norm)

        shap_values = explainer(X_teste_norm)

        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_teste_norm, feature_names=ultimos_niveis, max_display=10, show=False)
        plt.close()
        #plt.title(f'SHAP — {nome_par}')
        plt.tight_layout()
        
        save_path_shap = os.path.join(PASTA_PLOTS, model_name.replace('.joblib', '_shap.png'))
        plt.savefig(save_path_shap, dpi=300, bbox_inches='tight')
    
        #plt.show()

        
        print('-' * 70)


Controle vs. FBM
Número de táxons inicial: 532
Número de táxons pós-filtro: 172
Número de táxons pós-seleção: 108

--- Avaliando: Controle_vs_FBM_CatBoost.joblib ---
  Acurácia balanceada: 0.813
  PR AUC: 0.640
  F1-score: 0.745
----------------------------------------------------------------------

--- Avaliando: Controle_vs_FBM_EBM.joblib ---
  Acurácia balanceada: 0.791
  PR AUC: 0.696
  F1-score: 0.723
----------------------------------------------------------------------

--- Avaliando: Controle_vs_FBM_LogReg.joblib ---
  Acurácia balanceada: 0.811
  PR AUC: 0.675
  F1-score: 0.737
----------------------------------------------------------------------

--- Avaliando: Controle_vs_FBM_RandomForest.joblib ---
  Acurácia balanceada: 0.733
  PR AUC: 0.644
  F1-score: 0.653
----------------------------------------------------------------------

Controle vs. CFS
Número de táxons inicial: 532
Número de táxons pós-filtro: 166
Número de táxons pós-seleção: 79

--- Avaliando: Controle_vs_CF

<Figure size 640x480 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

### Metagenômica (Shotgun)

In [10]:
SEMENTE_ALEATORIA = 24
TAMANHO_TESTE = 0.15
PSEUDOCOUNT = 1
PASTA_MODELOS = 'modelos-shotgun/'
PASTA_PLOTS = 'modelos-shotgun/plots/' 

os.makedirs(PASTA_PLOTS, exist_ok=True)  

map_labels = {
    "controle": "Controle", 
    "endo": "Endometriose",
    "fbm": "Fibromialgia",
    "cfs": "SFC",
    "ibs": "SII",
    "cpp": "DPC",
    "enx": "Enxaqueca",
    "cpp_endo": "Endometriose + DPC"
}

data = pd.read_csv('taxonomia_shotgun_corrigida.tsv', sep='\t', index_col=0)
metadata = data[['group', 'study_id']].copy()

data["group"] = metadata.loc[data.index, "group"]

pares_separados = [
    ("Controle vs. FBM", data.loc[~data["group"].isin(['enx', 'cfs'])]),
    ("Controle vs. CFS", data.loc[~data["group"].isin(['enx', 'fbm'])]),
    ("Controle vs. ENX", data.loc[~data["group"].isin(['fbm', 'cfs'])]),
]

for nome_par, dados in pares_separados:
    print("\n" + "="*70)
    print(f"{nome_par}")
    print("="*70)

    labels_originais = dados["group"].values  

    y = (labels_originais != "controle").astype(int)
    
    dados = dados.drop(columns=['group', 'study_id', 'disease_group', 'batch'])
    
    nomes_taxons = dados.columns
    
    print(f'Número de táxons inicial: {len(dados.columns)}')
    
    nomes_para_plotar = [map_labels[label] for label in labels_originais]
    
    dados_rel = dados.div(dados.sum(axis=1), axis=0)
    keep_taxa = (dados_rel >= 0.0001).sum(axis=0) >= (0.60 * dados_rel.shape[0])
    dados = dados.loc[:, keep_taxa]
    
    X = dados.values
    
    print(f'Número de táxons pós-filtro: {len(dados.columns)}')
    
    X_treino, X_teste, y_treino, y_teste = train_test_split(
        X, y, test_size=TAMANHO_TESTE, random_state=SEMENTE_ALEATORIA, stratify=y
    )

    X_treino_clr = clr(X_treino + PSEUDOCOUNT)
    X_teste_clr = clr(X_teste + PSEUDOCOUNT)
    
    grupos = np.unique(y_treino)
    g1 = X_treino_clr[y_treino == grupos[0]]
    g2 = X_treino_clr[y_treino == grupos[1]]
    
    pvals = []
    
    for i in range(X_treino_clr.shape[1]):
        try:
            stat, p = mannwhitneyu(g1[:, i], g2[:, i], alternative="two-sided")
        except ValueError:
            p = 1.0
        pvals.append(p)
        
    pvals = np.array(pvals)
    
    _, pvals_corr, _, _ = multipletests(pvals, method="fdr_bh")
    
    n_features_desejado = int(0.3 * X_treino_clr.shape[0])
    idx_top = np.argsort(pvals_corr)[:n_features_desejado]
    
    X_treino_sel = X_treino_clr[:, idx_top]
    X_teste_sel  = X_teste_clr[:, idx_top]
    
    dados = dados.iloc[:, idx_top]

    print(f'Número de táxons pós-seleção: {len(dados.columns)}')

    normalizador = StandardScaler()
    X_treino_norm = normalizador.fit_transform(X_treino_sel)
    X_teste_norm = normalizador.transform(X_teste_sel) 

    nome_base_arquivo = nome_par.replace(' vs. ', '_vs_')
    padrao_busca = f"{PASTA_MODELOS}{nome_base_arquivo}*.joblib"
    modelos_encontrados = glob.glob(padrao_busca)

    for file_path in modelos_encontrados:
        model_name = os.path.basename(file_path) 
        print(f"\n--- Avaliando: {model_name} ---")

        modelo = load(file_path)

        y_pred = modelo.predict(X_teste_norm)
        y_proba = modelo.predict_proba(X_teste_norm)

        print(f"  Acurácia: {accuracy_score(y_teste, y_pred):.3f}")
        print(f"  ROC AUC: {roc_auc_score(y_teste, y_proba[:, 1]):.3f}")
        print(f"  F1-score: {f1_score(y_teste, y_pred):.3f}")
        
        plt.figure(figsize=(7, 6))
        ax1 = plt.gca()

        ConfusionMatrixDisplay.from_estimator(
            modelo,
            X_teste_norm,
            y_teste,
            ax=ax1,
            display_labels=[
                map_labels["controle"],
                map_labels[np.unique(labels_originais[labels_originais != "controle"])[0]]
            ],
            cmap='RdPu'
        )

        ax1.set_xlabel('Predito', fontsize=12)
        ax1.set_ylabel('Verdadeiro', fontsize=12)
        
        plt.tight_layout()
        
        save_path_cm = os.path.join(PASTA_PLOTS, model_name.replace('.joblib', '_confusion_matrix.png'))
        plt.savefig(save_path_cm, dpi=300, bbox_inches='tight')

        #plt.show()
        plt.close()

        plt.figure(figsize=(7, 6))
        ax2 = plt.gca()
        
        nome_controle = map_labels["controle"]
        nome_doenca = map_labels[np.unique(labels_originais[labels_originais != "controle"])[0]]

        RocCurveDisplay.from_estimator(
            modelo,
            X_teste_norm,
            y_teste,
            ax=ax2,
            name=f'{nome_controle} vs {nome_doenca}',
            color='#C04ABC'
        )

        ax2.plot([0, 1], [0, 1], 'k--', label='Chance (AUC = 0.5)')
        ax2.set_xlabel('Taxa de Falsos Positivos (1 - Especificidade)', fontsize=12)
        ax2.set_ylabel('Taxa de Verdadeiros Positivos (Sensibilidade)', fontsize=12)
        ax2.legend(loc='lower right')

        plt.tight_layout()
        
        save_path_pr = os.path.join(PASTA_PLOTS, model_name.replace('.joblib', '_roc.png'))
        plt.savefig(save_path_pr, dpi=300, bbox_inches='tight')

        #plt.show()
        plt.close()

        if hasattr(modelo, "feature_importances_"):  
            explainer = shap.TreeExplainer(modelo)
        else:
            explainer = shap.Explainer(modelo, X_treino_norm)

        shap_values = explainer(X_teste_norm)

        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_teste_norm, feature_names=nomes_taxons, max_display=10, show=False)
        plt.close()
        #plt.title(f'SHAP — {nome_par}')
        plt.tight_layout()

        save_path_shap = os.path.join(PASTA_PLOTS, model_name.replace('.joblib', '_shap.png'))
        plt.savefig(save_path_shap, dpi=300, bbox_inches='tight')

        #plt.show()
        
        print('-' * 70)


Controle vs. FBM
Número de táxons inicial: 1982
Número de táxons pós-filtro: 234
Número de táxons pós-seleção: 24

--- Avaliando: Controle_vs_FBM_CatBoost.joblib ---
  Acurácia: 0.933
  ROC AUC: 1.000
  F1-score: 0.933
----------------------------------------------------------------------

--- Avaliando: Controle_vs_FBM_EBM.joblib ---
  Acurácia: 1.000
  ROC AUC: 1.000
  F1-score: 1.000
----------------------------------------------------------------------

--- Avaliando: Controle_vs_FBM_LogReg.joblib ---
  Acurácia: 0.933
  ROC AUC: 1.000
  F1-score: 0.923
----------------------------------------------------------------------

--- Avaliando: Controle_vs_FBM_RandomForest.joblib ---
  Acurácia: 0.933
  ROC AUC: 1.000
  F1-score: 0.933
----------------------------------------------------------------------

Controle vs. CFS
Número de táxons inicial: 1982
Número de táxons pós-filtro: 240
Número de táxons pós-seleção: 23

--- Avaliando: Controle_vs_CFS_CatBoost.joblib ---
  Acurácia: 0.867

<Figure size 640x480 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>